# 🧪 LocateAnything-3B · Đếm SẢN PHẨM theo VẠCH (supervision)

Đếm **cắt vạch** (LineZone) trên 3 video thật, gán nhãn bằng **supervision**
(box bo góc + nhãn + trace + vạch IN/OUT). Xuất **video annotate** xem inline.

| Video | Đếm | Vạch |
|---|---|---|
| `packages_rollers.mp4` | package (con lăn) | ngang y≈0.65 |
| `packages_belt.mp4` | package (có nhãn) | ngang y≈0.60 |
| `tomatoes_sorting.mp4` | cà chua | ngang y≈0.72 |

**Vì sao trước đếm ra 0:** `stride` lớn làm ByteTrack mất dấu vật giữa các frame,
cửa sổ quá ngắn, và prompt mô tả dài khiến model detect thưa. Bản này sửa:
**stride=1** (bám track liên tục) · **nhiều frame hơn** · **prompt CHUNG** (box/tomato)
để detect dày → đếm được. Cell cuối test riêng vài **query KHÓ** (chỉ đo có detect ra không).

*Không dùng polygon — chỉ đếm theo VẠCH.*

In [ ]:
# ⚙️ Cài đặt: clone repo (kèm video) + phụ thuộc + kiểm tra GPU
import os, sys, subprocess

def sh(*a):
    print("$", " ".join(a)); subprocess.run(list(a), check=True)

if os.path.isdir("/kaggle/working"): WORK = "/kaggle/working"
elif os.path.isdir("/content"):      WORK = "/content"
else:                                 WORK = os.getcwd()
os.chdir(WORK); print("WORK =", WORK)

BRANCH = "claude/locate-anything-test-suite-xwju2f"
URL    = "https://github.com/nguyendinhhuyht20032004-ai/VisionOS.git"
REPO   = os.path.join(WORK, "VisionOS")
if not os.path.isdir(os.path.join(REPO, ".git")):
    sh("git", "clone", "--depth", "1", "--branch", BRANCH, URL, REPO)
else:
    sh("git", "-C", REPO, "fetch", "--depth", "1", "origin", BRANCH)
    sh("git", "-C", REPO, "reset", "--hard", "origin/" + BRANCH)

CODE = os.path.join(REPO, "VisionOS")          # code + sample_videos/ ở đây
os.chdir(CODE); sys.path.insert(0, CODE)
print("CODE =", CODE)

subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "transformers==4.57.1", "accelerate", "supervision",
                "eva-decord", "lmdb"], check=False)

for m in [m for m in list(sys.modules)
          if m.split(".")[0] in ("la_counting", "recognition", "run_la_conveyor")]:
    del sys.modules[m]

try:
    import torch
    print("CUDA:", torch.cuda.is_available(),
          torch.cuda.get_device_name(0) if torch.cuda.is_available() else "")
except Exception as e:
    print("torch:", e)

In [ ]:
# 🎯 Cấu hình 3 video (VẠCH + prompt) + xem trước vạch (không cần GPU)
import cv2, numpy as np, matplotlib.pyplot as plt

VID = os.path.join(CODE, "sample_videos")
VIDEOS = [
  {"name": "Package · con lăn", "path": os.path.join(VID, "packages_rollers.mp4"),
   "orient": "horizontal", "line_pos": 0.65,
   "query": "cardboard box",                       # prompt CHUNG để ĐẾM (detect dày)
   "hard":  ["a sealed cardboard box", "kiện hàng carton"]},   # query KHÓ (test detect)
  {"name": "Package · có nhãn", "path": os.path.join(VID, "packages_belt.mp4"),
   "orient": "horizontal", "line_pos": 0.60,
   "query": "cardboard box",
   "hard":  ["a package with a shipping label", "a package with a barcode"]},
  {"name": "Cà chua · phân loại", "path": os.path.join(VID, "tomatoes_sorting.mp4"),
   "orient": "horizontal", "line_pos": 0.72,
   "query": "tomato",
   "hard":  ["a ripe red tomato", "cà chua"]},
]

# NÚM tốc độ / chất lượng (LA chậm — tăng để đếm kỹ hơn, giảm cho nhanh)
PROC_WIDTH     = 768    # res cao hơn = detect tốt hơn (trước 640 quá nhỏ)
MAX_FRAMES     = 45     # nhiều frame hơn để vật kịp qua vạch
STRIDE         = 1      # QUAN TRỌNG: =1 để ByteTrack bám track (trước 3 → mất dấu)
MAX_NEW_TOKENS = 512    # nhiều token = detect dày hơn (trước 256 quá ít)

def _prev(v, frac=0.5):
    cap = cv2.VideoCapture(v["path"]); n = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.set(cv2.CAP_PROP_POS_FRAMES, int(n * frac)); ok, fr = cap.read(); cap.release()
    if not ok: return None
    h, w = fr.shape[:2]; y = int(v["line_pos"] * h)
    cv2.line(fr, (0, y), (w, y), (0, 0, 255), 3)
    return cv2.cvtColor(fr, cv2.COLOR_BGR2RGB)

fig, axes = plt.subplots(1, len(VIDEOS), figsize=(16, 4))
for ax, v in zip(np.atleast_1d(axes), VIDEOS):
    im = _prev(v)
    if im is not None: ax.imshow(im)
    ax.set_title(v["name"] + "\n(vạch đỏ)"); ax.axis("off")
plt.tight_layout(); plt.show()
print("→ Vạch đỏ cắt ngang dòng vật chưa? Lệch thì sửa line_pos rồi chạy lại cell này.")

In [ ]:
# 🧠 Nạp LocateAnything-3B MỘT LẦN (lần đầu tải ~6GB, hơi lâu)
from run_la_conveyor import build_fast_detector
detector = build_fast_detector("nvidia/LocateAnything-3B",
                               max_new_tokens=MAX_NEW_TOKENS, iou=0.5, max_boxes=80)
print("✅ Model sẵn sàng.")

In [ ]:
# ▶️ ĐẾM theo VẠCH (prompt chung) → xuất video annotate + xem inline
import time, subprocess
from IPython.display import Video, display
from run_la_conveyor import run_video

OUT = os.path.join(WORK, "out_annot"); os.makedirs(OUT, exist_ok=True)
print(f"{'video':22}{'prompt':16}{'IN':>4}{'OUT':>5}{'tổng':>6}{'det/fr':>8}{'giây':>7}")
print("-" * 69)
for v in VIDEOS:
    q, stem = v["query"], os.path.splitext(os.path.basename(v["path"]))[0]
    mp4 = os.path.join(OUT, f"{stem}.mp4")
    t0 = time.time()
    r = run_video(detector, v["path"], q, orient=v["orient"], line_pos=v["line_pos"],
                  proc_width=PROC_WIDTH, max_frames=MAX_FRAMES, stride=STRIDE, save_video=mp4)
    print(f"{v['name'][:21]:22}{q[:15]:16}{r.in_count:>4}{r.out_count:>5}"
          f"{r.total_crossings:>6}{r.avg_detections:>8.1f}{time.time()-t0:>7.0f}")
    # đổi sang H.264 để xem inline (mp4v không phát được trên trình duyệt)
    h264 = mp4.replace(".mp4", "_h264.mp4")
    subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", mp4,
                    "-vcodec", "libx264", "-pix_fmt", "yuv420p", h264], check=False)
    display(Video(h264 if os.path.exists(h264) else mp4, embed=True, width=560))
print("📁 Video annotate lưu tại:", OUT, "(cả .mp4 gốc và *_h264.mp4)")

In [ ]:
# 🔬 (tuỳ chọn) TEST vài QUERY KHÓ — chỉ đo model có DETECT ra không (nhanh)
import time
from run_la_conveyor import run_video

print(f"{'video':22}{'query khó':40}{'det/fr':>8}{'giây':>7}")
print("-" * 78)
for v in VIDEOS:
    for q in v.get("hard", []):
        t0 = time.time()
        r = run_video(detector, v["path"], q, orient=v["orient"], line_pos=v["line_pos"],
                      proc_width=PROC_WIDTH, max_frames=8, stride=3)   # ít frame = nhanh
        print(f"{v['name'][:21]:22}{q[:39]:40}{r.avg_detections:>8.1f}{time.time()-t0:>7.0f}")
print("det/fr>0 = model HIỂU mô tả và bắt được vật (điểm mạnh open-vocab).")

### Đọc kết quả
- **Cell ĐẾM** (prompt chung box/tomato): `tổng` = số vật cắt vạch (IN+OUT). Muốn nhiều
  hơn: tăng `MAX_FRAMES`. Vạch lệch: sửa `line_pos` (cell cấu hình).
- **Cell QUERY KHÓ**: `det/fr` > 0 nghĩa là LocateAnything hiểu mô tả (kể cả tiếng Việt)
  — đây là thứ YOLO COCO không làm được. Query khó thường detect THƯA nên không hợp để đếm,
  chỉ để chứng minh khả năng open-vocab.
- **Video**: `WORK/out_annot/<video>.mp4` (gốc, mp4v) và `<video>_h264.mp4` (xem inline / tải về).
- LocateAnything **chậm** (~vài giây/frame). `stride=1` là bắt buộc để đếm đúng — đừng tăng.